# Music Genre Classifier - CNN

## Reading/Importing Dataset
Here the images are all added to a folder so they can be easily accessed.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shutil
from lib.data_utils import all_images, load_data
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from lib.layers import *


csv_path = "Data/features_3_sec.csv"
df = pd.read_csv(csv_path)

img_dir_path = "Data/all_images"

img_size = (128, 128)

all_images() # creates a folder of all images (regardless of initial folder)

2025-03-26 13:49:29.616300: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Preprocessing the Data
This is making all of the spectrograms (9980) into arrays with size (128, 128) with 1 channel.
We made it black and white instead of with multiple colors for ease of training. There will be one label to go with all of the images (so y should have shape 9980).

In [2]:
labels = sorted(df['label'].unique())
label_to_index = {label: idx for idx, label in enumerate(labels)}
df['label_idx'] = df['label'].map(label_to_index)

X, y = load_data(df, img_dir_path, img_size) # loads all of the spectrogram and label data
print("Data successfully loaded.")

print(X.shape)
print(y.shape)

Data successfully loaded.
(9980, 128, 128, 1)
(9980,)


## Spliting up Testing and Training Data
The model is going to be trained on 80% of the data and tested on 20% of the data. We should see a generally even split between all genres for both training and testing. This would mean that there should be about 798 images per genre for the training (because 7982/10 = 789.2) and 200 images per genre for testing (1998/10 = 199.8).

In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label_idx'], random_state=42)
X_train, y_train = load_data(train_df, img_dir_path, img_size)
X_test, y_test = load_data(test_df, img_dir_path, img_size)

X_train_shape = X_train.shape
X_test_shape = X_test.shape

y_train_shape = y_train.shape
y_test_shape = y_test.shape

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

print("\nNumber of Training Images per Genre:")
print(train_df['label_idx'].value_counts().sort_index())

print("\nNumber of Testing Images per Genre:")
print(test_df['label_idx'].value_counts().sort_index())

X_train Shape: (7982, 128, 128, 1)
X_test Shape: (1998, 128, 128, 1)

Number of Training Images per Genre:
0    800
1    799
2    798
3    799
4    798
5    800
6    800
7    800
8    800
9    798
Name: label_idx, dtype: int64

Number of Testing Images per Genre:
0    200
1    199
2    199
3    200
4    200
5    200
6    200
7    200
8    200
9    200
Name: label_idx, dtype: int64


## Layers to cover wide ranges of music:
1. Vertical patterns (like beat from drum)
2. Horizontal patterns (long, held notes)

We might need more layers but for this week we should stick to just a few.

In [4]:
v_filter = np.array([[[[0, 1, 0],
                        [0, 1, 0],
                         [0, 1, 0]]]])

h_filter = np.array([[[[0, 0, 0], 
            [1, 1, 1], 
            [0, 0, 0]]]]) # horizontal filter

d_filter = np.array([[[[1, 0, 0], 
            [0, 1, 0], 
            [0, 0, 1]]]]) # diagonal filter

b = np.array([0])

## Applying the Filters to the Model
X_shape = (N, H, W, C); N = Number of samples, H = height of the image, W = width of the image, C = channels

w_shape (filter) = (F, C, HH, WW); F = number of filters being used, C = number of channels, HH = height of filter, WW = weight of filter

In [5]:
print("Training Data Shape: ", X_train_shape)
print("Vertical Filter Shape: ", v_filter.shape)
print("Horizontal Filter Shape: ", h_filter.shape)
print("Diagonal Filter Shape: ", d_filter.shape)

Training Data Shape:  (7982, 128, 128, 1)
Vertical Filter Shape:  (1, 1, 3, 3)
Horizontal Filter Shape:  (1, 1, 3, 3)
Diagonal Filter Shape:  (1, 1, 3, 3)


In [7]:
conv_param = {'stride': 1, 'pad': 1}
pool_param = {'pool_width': 2, 'pool_height': 2, 'stride': 2}

# First filter - vertical filtering
out, _ = conv_forward(X_train, v_filter, b, conv_param)
print(out)

[[[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 ...


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. 

In [9]:
out, relu_cache = relu_forward(out)
print(out)

[[[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 ...


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. ... 6. 6. 4.]
   [2. 3. 3. ... 3. 3. 2.]]]


 [[[6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   [6. 9. 9. ... 9. 9. 6.]
   ...
   [6. 9. 9. ... 9. 9. 6.]
   [4. 6. 6. 

In [11]:
out, pool_cache = max_pool_forward(out, pool_param)
print(out)

[[[[9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   ...
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [6. 6. 6. ... 6. 6. 6.]]]


 [[[9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   ...
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [6. 6. 6. ... 6. 6. 6.]]]


 [[[9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   ...
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [6. 6. 6. ... 6. 6. 6.]]]


 ...


 [[[9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   ...
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [6. 6. 6. ... 6. 6. 6.]]]


 [[[9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   ...
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [6. 6. 6. ... 6. 6. 6.]]]


 [[[9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. ... 9. 9. 9.]
   ...
   [9. 9. 9. ... 9. 9. 9.]
   [9. 9. 9. 

In [12]:
print(out.shape)

(7982, 1, 64, 64)


In [13]:
# Second filter - horizontal filtering
out, _ = conv_forward(X_train, h_filter, b, conv_param)
out, relu_cache = relu_forward(out)
out, pool_cache = max_pool_forward(out, pool_param)

# Third filter - diagonal filtering
out, _ = conv_forward(X_train, d_filter, b, conv_param)
out, relu_cache = relu_forward(out)
out, pool_cache = max_pool_forward(out, pool_param)

In [14]:
# Flatten the output
out_flat = out.reshape(out.shape[0], -1)  # shape: (N, C*H*W)

In [21]:
# Initialize weights and biases
W_dense = np.random.randn(out_flat.shape[1], 10) * 0.01
b_dense = np.zeros(10)

# Forward pass
scores = np.dot(out_flat, W_dense) + b_dense
print(scores.shape)

(7982, 10)


In [24]:
def softmax(x):
    exp_scores = np.exp(x - np.max(x, axis=1, keepdims=True))  # stability
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

probs = softmax(scores)
print(probs)

[[0.15997034 0.00528716 0.23378209 ... 0.06416586 0.18340655 0.02923124]
 [0.14062174 0.00406396 0.23386291 ... 0.06834576 0.27257309 0.0317037 ]
 [0.12535745 0.0043053  0.24190321 ... 0.05716903 0.22712468 0.04001504]
 ...
 [0.12085087 0.00532303 0.22052002 ... 0.0601836  0.22556313 0.04121657]
 [0.15842212 0.00407683 0.26105354 ... 0.06978956 0.21195193 0.02766079]
 [0.14309711 0.00364635 0.27417093 ... 0.05665189 0.23335028 0.03441107]]


In [31]:
genre_labels = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

y_pred = np.argmax(probs, axis=1)

# Print first N predictions and their actual labels
N = 10  # or however many you want to inspect

count = 0
for i in range(0, len(y_pred)):
    pred_genre = genre_labels[y_pred[i]]
    true_genre = genre_labels[y_train[i]]
    print(f"Sample {i}: predicted → {pred_genre}, actual → {true_genre}")
    if pred_genre == true_genre:
        count += 1

Sample 0: predicted → country, actual → hiphop
Sample 1: predicted → reggae, actual → metal
Sample 2: predicted → country, actual → pop
Sample 3: predicted → country, actual → metal
Sample 4: predicted → reggae, actual → country
Sample 5: predicted → country, actual → rock
Sample 6: predicted → country, actual → classical
Sample 7: predicted → country, actual → blues
Sample 8: predicted → country, actual → reggae
Sample 9: predicted → country, actual → pop
Sample 10: predicted → country, actual → disco
Sample 11: predicted → country, actual → rock
Sample 12: predicted → reggae, actual → classical
Sample 13: predicted → country, actual → reggae
Sample 14: predicted → country, actual → disco
Sample 15: predicted → country, actual → disco
Sample 16: predicted → country, actual → country
Sample 17: predicted → reggae, actual → jazz
Sample 18: predicted → country, actual → classical
Sample 19: predicted → country, actual → blues
Sample 20: predicted → country, actual → jazz
Sample 21: predi

In [32]:
print(count/len(y_pred))

0.10085191681282886


## Next Steps:
1. Create functions for convolution layers, max pooling layers, etc.
2. Determine when forward and backpropagation are necessary (need backpropagation for training to determine error).